# Cascadia Cryptid Sighting Survey — a correct analysis

Companion to `trap_demos.ipynb`, which shows what goes wrong. This notebook does the
analysis the way it should be done, in the order the decisions actually arise:

1. establish what a row is and what the quantity of interest is
2. choose the outcome distribution from the mean-variance relationship, not by habit
3. diagnose the collinear block before fitting anything interpretive
4. let the data choose functional form for the continuous predictor
5. test the categorical predictor as a block
6. report rate ratios with intervals, and say plainly what is not identified
7. check the fit with residuals that are valid for count data

Figures follow the project visualisation conventions: Okabe-Ito palette, Arial,
SVG in `figures/` for editing, axis labels carrying units, legends top right.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import analysis_plots as ap
from analysis_plots import OI

ap.set_style()
d = pd.read_csv("cryptid_sightings.csv")
print(d.shape)
d.head()

## 1. What is a row, and what are we estimating?

A row is one ranger district in one month, and those months are **not the same size**:
exposure runs from 4 to 211 hundred visitor-hours. Reports are filed by visitors, so
the count scales with exposure almost by construction. The estimand is therefore a
**rate** — reports per 100 visitor-hours — and exposure belongs in the model as an
offset, not as a predictor competing for credit.

In [ ]:
d["rate"] = d.sightings / d.visitor_hundred_hours
d["log_exposure"] = np.log(d.visitor_hundred_hours)

print("exposure range: %d to %d (median %d), a %.0f fold spread"
      % (d.visitor_hundred_hours.min(), d.visitor_hundred_hours.max(),
         d.visitor_hundred_hours.median(),
         d.visitor_hundred_hours.max() / d.visitor_hundred_hours.min()))
print("overall rate: %.3f reports per 100 visitor hours"
      % (d.sightings.sum() / d.visitor_hundred_hours.sum()))
print("correlation of exposure with vendor B canopy: %.3f"
      % d.log_exposure.corr(d.canopy_vendor_b))

Exposure is correlated with one of the canopy columns, so omitting it would not just
cost precision — it would bias the canopy estimates. Look at the distributions before
choosing a model.

In [ ]:
ap.plot_outcome(d);

## 2. Choosing the outcome distribution

Poisson regression assumes variance equals mean. Check that against the data rather
than assuming it: bin observations by fitted mean and compare the within-bin variance
to the Poisson and negative binomial expectations.

In [ ]:
FORMULA_LINEAR = ("sightings ~ canopy_mean_s + canopy_diff + beer_sales_z"
                  " + I(beer_sales_z ** 2) + C(ranger_district)")

# Reparameterise the vendor block before fitting: see section 3.
d["canopy_mean_s"] = ((d.canopy_vendor_a + d.canopy_vendor_b) / 2 - 50) / 10
d["canopy_diff"] = d.canopy_vendor_a - d.canopy_vendor_b

# Negative binomial with the dispersion estimated, not assumed.
nb = smf.negativebinomial(FORMULA_LINEAR, d, offset=d.log_exposure).fit(disp=0)
alpha = nb.params["alpha"]
print("estimated alpha = %.3f  ->  implied size theta = %.2f" % (alpha, 1 / alpha))

d["fitted_mu"] = nb.predict(d, offset=d.log_exposure)
pois = smf.glm(FORMULA_LINEAR, d, family=sm.families.Poisson(),
               offset=d.log_exposure).fit()
print("Poisson Pearson chi2/df = %.1f (should be near 1)"
      % ((pois.resid_pearson ** 2).sum() / pois.df_resid))
print("raw variance / mean = %.1f" % (d.sightings.var() / d.sightings.mean()))

In [ ]:
ap.plot_mean_variance(d, theta=1 / alpha);

The variance grows quadratically with the mean, so the negative binomial is the right
family and the Poisson standard errors would be far too small. Note that the dispersion
is **estimated** here (alpha = 0.55, i.e. size near 1.8; the generating value was 1.6) rather than taken as known.

## 3. The collinear block: diagnose, then reparameterise

Vendor A and vendor B measure the same canopy. Dropping one is the common reflex and the
wrong move: the survivor's coefficient becomes an uninterpretable blend. The honest
alternative is to **rotate the pair into quantities that mean something** —
their average (the canopy signal) and their difference (vendor disagreement) — which
leaves the model fit unchanged but makes identification visible in the output.

In [ ]:
X = sm.add_constant(d[["canopy_vendor_a", "canopy_vendor_b",
                       "beer_sales_z", "ranger_district"]])
vif = {c: 1 / (1 - sm.OLS(X[c], sm.add_constant(X.drop(columns=[c]))).fit().rsquared)
       for c in X.columns if c != "const"}
print("VIF (original parameterisation):",
      {k: round(v, 1) for k, v in vif.items()})

X2 = sm.add_constant(d[["canopy_mean_s", "canopy_diff", "beer_sales_z", "ranger_district"]])
vif2 = {c: 1 / (1 - sm.OLS(X2[c], sm.add_constant(X2.drop(columns=[c]))).fit().rsquared)
        for c in X2.columns if c != "const"}
print("VIF (mean/difference parameterisation):",
      {k: round(v, 1) for k, v in vif2.items()})

In [ ]:
ap.plot_collinearity(d);

The rotation removes the collinearity entirely: the average and the difference are
orthogonal by construction. The difference has an SD of about 2.4 percentage points
and no reason to affect sightings — it is contractor noise. If it comes back
uninformative, averaging the two vendors is justified by the data as well as by sense.

## 4. Functional form for beer sales

Do not assume linearity. Compare a linear term against a quadratic by likelihood ratio,
and plot the empirical log rate by bin so the shape is visible rather than inferred.

In [ ]:
def lr_test(small, large, df):
    """Likelihood ratio test between nested fits."""
    stat = 2 * (large.llf - small.llf)
    return stat, stats.chi2.sf(stat, df)

no_quad = smf.negativebinomial(
    "sightings ~ canopy_mean_s + canopy_diff + beer_sales_z + C(ranger_district)",
    d, offset=d.log_exposure).fit(disp=0)
stat, p = lr_test(no_quad, nb, 1)
print("quadratic beer term: LR chi2 = %.1f on 1 df, p = %.2g" % (stat, p))
print("minimum of the fitted curve at z = %.2f"
      % (-nb.params["beer_sales_z"] / (2 * nb.params["I(beer_sales_z ** 2)"])))

In [ ]:
ap.plot_beer_shape(d, nb);

The U shape is real and large: rates are high at both ends of the beer-sales
distribution and lowest just above the mean. A linear fit would report a confident
monotone decline and get the story backwards at the top of the range.

## 5. District as a block, not a slope

`ranger_district` arrives as integers 1-5, but the codes are labels. Test the whole
factor with a single 4-df likelihood ratio test rather than reading a per-level
p-value, and never fit it as a continuous slope.

In [ ]:
no_district = smf.negativebinomial(
    "sightings ~ canopy_mean_s + canopy_diff + beer_sales_z + I(beer_sales_z ** 2)",
    d, offset=d.log_exposure).fit(disp=0)
stat, p = lr_test(no_district, nb, 4)
print("district factor: LR chi2 = %.1f on 4 df, p = %.2g" % (stat, p))

as_slope = smf.negativebinomial(
    "sightings ~ canopy_mean_s + canopy_diff + beer_sales_z + I(beer_sales_z ** 2)"
    " + ranger_district", d, offset=d.log_exposure).fit(disp=0)
print("same column as a continuous slope: beta = %.3f, p = %.2f"
      % (as_slope.params["ranger_district"], as_slope.pvalues["ranger_district"]))

A strongly significant block effect becomes a null result the moment the labels are
treated as magnitudes. That is the single cheapest mistake available in this dataset.

## 6. Report rate ratios, and state what is not identified

In [ ]:
LABELS = {
    "canopy_mean_s": "Canopy, vendor average (+10 pp)",
    "canopy_diff": "Canopy, vendor disagreement (+1 pp)",
    "beer_sales_z": "Beer sales (linear term, +1 SD)",
    "I(beer_sales_z ** 2)": "Beer sales (quadratic term)",
    "C(ranger_district)[T.2]": "Sasquatch Flats vs Mt. Humptulips",
    "C(ranger_district)[T.3]": "Grey Owl Basin vs Mt. Humptulips",
    "C(ranger_district)[T.4]": "Cougar Creek vs Mt. Humptulips",
    "C(ranger_district)[T.5]": "Lake Quinault vs Mt. Humptulips",
}
ci = nb.conf_int()
table = pd.DataFrame({
    "rr": np.exp(nb.params[list(LABELS)]),
    "lo": np.exp(ci.loc[list(LABELS), 0]),
    "hi": np.exp(ci.loc[list(LABELS), 1]),
    "p": nb.pvalues[list(LABELS)],
}).set_axis([LABELS[k] for k in LABELS])
table["unidentified"] = table.index.str.contains("disagreement")
table.round(3)

In [ ]:
ap.plot_coefficients(table);

The vendor disagreement term is flagged in a different colour because its interval is
wide and centred near 1: the data cannot resolve it, which is the expected result if
vendor disagreement is measurement noise. Having established that, the defensible
summary of the canopy effect is the **average of the two vendors**, with the
disagreement term either retained as an uninformative nuisance or dropped on the
strength of its own test — not a coefficient from a model where one vendor was
deleted to satisfy a VIF rule.

What must be said in the write-up, and cannot be read off a coefficient table:
the individual vendor slopes are **not separately estimable** at r = 0.974. Any
report that assigns an effect to vendor A rather than vendor B is describing the
contractors' measurement error, not the forest.

In [ ]:
# The same point in the original parameterisation, for the record.
orig = smf.negativebinomial(
    "sightings ~ a_s + b_s + beer_sales_z + I(beer_sales_z ** 2) + C(ranger_district)",
    d.assign(a_s=(d.canopy_vendor_a - 50) / 10, b_s=(d.canopy_vendor_b - 50) / 10),
    offset=d.log_exposure).fit(disp=0)
print(pd.DataFrame({"est": orig.params[["a_s", "b_s"]],
                    "se": orig.bse[["a_s", "b_s"]],
                    "p": orig.pvalues[["a_s", "b_s"]]}).round(3))
print("\nIdentical fit: log-likelihood %.3f vs %.3f" % (orig.llf, nb.llf))
print("The rotation changes the coordinates, not the model.")

## 7. Fit diagnostics for count data

Pearson or deviance residuals are not approximately normal for counts, so they are hard
to read. Randomised quantile residuals are exactly standard normal under a correctly
specified model, which makes a QQ plot interpretable.

In [ ]:
def randomised_quantile_residuals(y, mu, alpha, seed=1):
    """Dunn and Smyth residuals for an NB2 fit: standard normal if the model is right."""
    size = 1 / alpha
    prob = size / (size + mu)
    lower = stats.nbinom.cdf(y - 1, size, prob)
    upper = stats.nbinom.cdf(y, size, prob)
    u = np.random.default_rng(seed).uniform(lower, upper)
    return stats.norm.ppf(np.clip(u, 1e-10, 1 - 1e-10))

d["rqr"] = randomised_quantile_residuals(d.sightings, d.fitted_mu, alpha)
print("mean %.3f, SD %.3f (expect 0 and 1)" % (d.rqr.mean(), d.rqr.std()))
print("Shapiro-Wilk p = %.2f" % stats.shapiro(d.rqr).pvalue)

In [ ]:
ap.plot_diagnostics(d);

## 8. What the analysis concludes

- Reports scale with visitor exposure, so everything is reported as a rate per 100
  visitor-hours.
- The outcome is strongly overdispersed (variance about 52 times the mean, estimated
  alpha of 0.55); a Poisson fit would understate standard errors roughly fourfold.
- Beer sales have a U-shaped relationship with the report rate, minimum just above the
  mean, significant on a 1-df test.
- Ranger district is a strong block effect on a 4-df test and vanishes entirely if its
  integer codes are treated as a magnitude.
- Canopy cover is summarised by the vendor average. The two vendors' individual slopes
  are not separately identified and are not reported as separate effects; the vendor
  disagreement term is uninformative, consistent with contractor noise.
- Randomised quantile residuals are consistent with the fitted model, with no trend in
  mean or spread and no evidence of unmodelled structure.

The ground truth in `simulate_cryptids.py` confirms each of these, including the one
that matters most: the vendor slopes really are +0.330 and -0.295, and no analysis of
these 500 rows could have recovered them separately.